## 0. Подготовка

In [1]:
import os

import pandas as pd

from solution_utils import (
    reaction_metrics, init_reaction_models, predict_reaction,
    read_dialogs, read_labels, compute_metrics, show_table
)
import solution_utils as su

In [2]:
TRAIN_PATH = os.environ.get("HW1_TRAIN", "hw1_train.jsonl")
DATA_PATH = os.environ.get("HW1_DATA", "hw1_train.jsonl")
LABELS_PATH = os.environ.get("HW1_LABELS", DATA_PATH)
PREDICTIONS_PATH = os.environ.get("HW1_PREDICTIONS", "predictions.csv")

LABELS_PATH = "./hidden_labels.jsonl"
DATA_PATH = "./hidden_test.jsonl"

dialogs = read_dialogs(DATA_PATH)
print("диалогов:", len(dialogs))

диалогов: 990


## Загрузка и обработка данных

In [3]:
dialogs = su.read_dialogs(DATA_PATH)

frame_all = su.to_frame(dialogs)
print("диалогов:", len(frame_all), " размечено по реакции:", frame_all.reaction.notna().sum())
df = frame_all
df[["dialog_id", "last_raw", "n_user", "reaction"]].head()
display(df)

/home/salexq/miniconda3/envs/ml/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator MultinomialNB from version 1.9.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/salexq/miniconda3/envs/ml/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator CountVectorizer from version 1.9.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


диалогов: 990  размечено по реакции: 0


,dialog_id,text,first_user,rest_user,all_user,last_asst_head,first_asst,raw_first,last_raw,n_user,len_last,len_asst,prev_raw,rest_raw,n_asst,len_asst_first,len_asst_last,group,reaction
0,t62f1ed47f1,what are the all subject relat and object trip...,what are the all subject relat and object trip...,what are the all subject relat and object trip...,what are the all subject relat and object trip...,here are the subject relat object triplet for ...,subject a girl relat sit on object a bench rea...,"What are the all (subject, relation, and objec...","What are the all (subject, relation, and objec...",6,1795,1295.833333,"What are the all (subject, relation, and objec...","What are the all (subject, relation, and objec...",6,1932,1379,whataretheallsubjectrelationandobjecttripletsi...,None
1,tdf1e1c85b5,bot do you know what input and output mean in ...,bot do you know what input and output mean in ...,so if i want to know the energi consumpt of el...,so if i want to know the energi consumpt of el...,you re welcom ! i m glad you found the explan ...,yes in the context of an ac adapt input and ou...,"bot, do you know what 'input' and 'output' mea...","thank you, bot. your explanation is quite easy...",9,61,1321.333333,"so, what does the low and high current rate me...","so, if i want to know the energy consumption o...",9,865,206,botdoyouknowwhatinputandoutputmeansinacadapter,None
2,t25e6a6a481,note frequenc map for the game of throne theme...,note frequenc map for the game of throne theme,write a program to play game of throne on a es...,write a program to play game of throne on a es...,here s a sampl code for play the pink panther ...,sorri as an ai languag model i don t have acce...,Note frequency mapping for the Game of thrones...,write a program to play pink panter theme for ...,6,53,1525.666667,write a program to play pink panter theme on a...,write a program to play game of thrones on a e...,6,119,1360,notefrequencymappingforthegameofthronestheme,None
3,tef3424a000,sarah is a year old girl who has been live wit...,sarah is a year old girl who has been live wit...,can you correct the typo of this ? the abov es...,can you correct the typo of this ? the abov es...,sure here is a refer for the parent,as an ai languag model i cannot fulli assess t...,Sarah is a 12-year-old girl who has been livin...,Can you give me the reference for the modellin...,6,53,1820.833333,That was emphasized in the SWAY in a way that ...,Can you correct the typos of this ? the above ...,6,529,42,sarahisa12yearoldgirlwhohasbeenlivingwithjuven...,None
4,tc70e26a773,what danc is marcus koch famous for as an ai l...,what danc is marcus koch famous for,what danc is marcus koch famous for killer boo...,what danc is marcus koch famous for killer boo...,thank you for the correct i was not awar of ma...,as an ai languag model i cannot find ani famou...,what dances is marcus koch famous for,He was world champion in boogie woogie in 1993,5,46,406.800000,and world champion boogiw woogie,what dances is marcus koch famous for killer b...,5,135,751,whatdancesismarcuskochfamousfor,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
985,t0b0d1ac488,сократ этот текс до предложен или чут меньш th...,сократ этот текс до предложен или чут меньш th...,предложен должн быт связа друг с друг ве этот ...,предложен должн быт связа друг с друг ве этот ...,привет ! несомнен программирован это важн и по...,engin have problem solv skill and mental acume...,сократи этот текс до 20 предложений или чуть м...,привет мой скилл это программирование ответь м...,6,254,1625.666667,создай большой текст из тех предложений можешь...,предложения должны быть связаны друг с другом ...,6,2256,1132,20thebenefitsofbeinganengineerinsightsfromelon...,None
986,t215f15d7a0,i am creat a system that detect when a user sa...,i am creat a system that detect when a user sa...,hypothet i am creat a system that detect when ...,hypothet i am creat a system that detect when ...,as an ai

In [4]:
reaction_models = init_reaction_models("./artifacts/reaction_models")
reaction_labels = read_labels(LABELS_PATH, "reaction") if LABELS_PATH else {}

reaction_predictions, reaction_metrics_tb = {}, {}
for name, model in reaction_models.items():
    reaction_predictions[name] = predict_reaction(model, df)
    if reaction_labels:
        reaction_metrics_tb[name] = compute_metrics(dialogs, reaction_predictions[name], reaction_labels, reaction_metrics)

show_table(pd.DataFrame(reaction_metrics_tb).T.round(3))

,macro_f1,macro_f1_bal,accuracy,balanced_acc,P_negative,R_negative,F1_negative,P_neutral,R_neutral,F1_neutral,P_positive,R_positive,F1_positive
hw1_dummy_most_frequent,0.167,0.167,0.333,0.333,0.000,0.000,0.000,0.333,1.000,0.500,0.000,0.000,0.000
hw1_linear_svm,0.442,0.442,0.464,0.464,0.438,0.515,0.474,0.419,0.667,0.515,0.896,0.209,0.339
hw1_linear_svm_my,0.523,0.523,0.530,0.530,0.501,0.524,0.513,0.457,0.733,0.563,0.948,0.333,0.493
hw1_logreg,0.456,0.456,0.474,0.474,0.438,0.524,0.477,0.433,0.667,0.525,0.874,0.230,0.365
hw1_logreg_my,0.541,0.541,0.542,0.542,0.492,0.588,0.536,0.478,0.664,0.556,0.899,0.376,0.530
hw1_nb,0.320,0.320,0.383,0.383,0.384,0.567,0.458,0.371,0.552,0.443,0.833,0.030,0.058
hw1_nb_my,0.478,0.478,0.490,0.490,0.450,0.594,0.512,0.436,0.615,0.510,0.977,0.261,0.411
hw2_gb_lsa,0.532,0.532,0.542,0.542,0.547,0.524,0.536,0.457,0.782,0.577,0.964,0.321,0.482
hw2_hgb_lsa,0.566,0.566,0.562,0.562,0.495,0.570,0.530,0.499,0.618,0.552,0.816,0.497,0.618
hw2_rf_lsa,0.566,0.566,0.561,0.561,0.507,0.524,0.516,0.476,0.664,0.554,0.862,0.494,0.628
